In [6]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import time

In [7]:
# 1. Dataset Preparation
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

# Normalize pixel values
normalization_layer = tf.keras.layers.Rescaling(1./255.)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Found 8000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [8]:
# 2. Function to build models
def build_model(base_model_class, name):
    # Load base model with ImageNet weights, exclude top
    base = base_model_class(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False  # Freeze base layers
    
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    # Binary classification output (1 neuron, sigmoid)
    predictions = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base.input, outputs=predictions)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [9]:
# 3. Train and Evaluate Loop
results = []
architectures = [
    (MobileNetV2, 'MobileNetV2'),
    (ResNet50, 'ResNet50'),
    (EfficientNetB0, 'EfficientNetB0')
]

for arch, name in architectures:
    print(f"\n--- Training {name} ---")
    model = build_model(arch, name)
    
    start_time = time.time()
    history = model.fit(train_ds, epochs=5, validation_data=test_ds, verbose=1)
    end_time = time.time()
    
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    duration = end_time - start_time
    
    results.append({
        'Model': name,
        'Train Acc': f"{train_acc:.2%}",
        'Val Acc': f"{val_acc:.2%}",
        'Time (s)': f"{duration:.1f}"
    })
    
    # Save the best model for Streamlit
    if name == 'MobileNetV2': # Assuming this wins
        model.save('best_model.h5')

print("\nTraining Complete.")


--- Training MobileNetV2 ---
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 159s 603ms/step - accuracy: 0.9563 - loss: 0.1197 - val_accuracy: 0.9810 - val_loss: 0.0527
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 176s 705ms/step - accuracy: 0.9796 - loss: 0.0538 - val_accuracy: 0.9815 - val_loss: 0.0453
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 220s 878ms/step - accuracy: 0.9811 - loss: 0.0459 - val_accuracy: 0.9825 - val_loss: 0.0429
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 225s 901ms/step - accuracy: 0.9834 - loss: 0.0402 - val_accuracy: 0.9845 - val_loss: 0.0413
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 244s 826ms/step - accuracy: 0.9872 - loss: 0.0342 - val_accuracy: 0.9865 - val_loss: 0.0407



--- Training ResNet50 ---
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 655s 3s/step - accuracy: 0.5355 - loss: 0.7019 - val_accuracy: 0.6195 - val_loss: 0.6654
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 782s 3s/step - accuracy: 0.5971 - loss: 0.6659 - val_accuracy: 0.6135 - val_loss: 0.6611
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 649s 3s/step - accuracy: 0.6114 - loss: 0.6576 - val_accuracy: 0.6080 - val_loss: 0.6601
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 640s 3s/step - accuracy: 0.6229 - loss: 0.6502 - val_accuracy: 0.6285 - val_loss: 0.6525
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 652s 3s/step - accuracy: 0.6158 - loss: 0.6498 - val_accuracy: 0.6285 - val_loss: 0.6502

--- Training EfficientNetB0 ---
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 272s 1s/step - accuracy: 0.5011 - loss: 0.6998 - val_accuracy: 0.5000 - val_loss: 0.6946
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 248s 991ms/step - accuracy: 0.4995 - loss: 0.6983 - val_accuracy: 0.5000 - val_loss: 0.6950
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 24